# PER Global Data Timeline

Every sample from every REV 11 log, in one queryable store.

The problem this solves: today a question like *"which testing sessions in May 
had a cell voltage sag?"* means downloading and parsing logs one at a time. 
There is no way to ask a question **across** the season.

### How it's built

| Layer | Where | Size | What it answers |
|---|---|---|---|
| `test_days` / `sessions` | Postgres | KB | *when did we test* |
| `variables` (catalog) | Postgres | MB | *what does the car log* |
| `session_var_stats` | Postgres | ~50 MB | **most questions** |
| `samples` | TimescaleDB hypertable, columnstore | ~4 bytes/row | waveform detail |

The trick is the third row. Per-session-per-variable summaries are small enough to scan instantly, so threshold questions never touch the billions of raw samples. Raw data is read only after the summary tier has narrowed things to a handful of sessions.

Two facts that shape everything:

- **The numeric variable ids in a log header are not stable** — not even between two REV 11 builds. The dotted C++ path (`bms.stack.mma.cellV.min`) is the only durable key, so ingest remaps every file's local ids onto a canonical catalog.
- **Timestamps are stored as absolute epoch microseconds**, not session-relative. That is what lets the hypertable prune by calendar date. A log's absolute start comes from its header line (UTC) — the filename is local Philadelphia time, and `pcm.startTime` is an RTC uptime counter that never got set.

In [1]:
import time

import plotly.graph_objects as go
import polars as pl

from perda.timeline.client import TimelineClient

pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(60)

tl = TimelineClient()
tl.overview()

test_day,sessions,minutes,rows,max_vars
date,i64,f64,"decimal[38,0]",i64
2026-05-01,19,107.0,113337517,1072
2026-05-02,32,93.0,106425663,1078
2026-05-11,30,91.0,101011178,1078
2026-05-12,27,105.0,117256102,1078
2026-05-13,43,227.0,212129257,1090
2026-05-14,34,214.0,239783691,1087
2026-05-18,42,248.0,274800491,1090
2026-05-20,26,209.0,250133907,1096
2026-05-26,11,131.0,156506165,1098


## 1. The variable catalog

Built from the header block of every log. `dtype` is inferred from observed values and only ever widens (`bool` → `int` → `float`), so a variable that happens to sit in {0,1} for one short session cannot be mislabelled forever.

In [2]:
print('catalog size:', tl.sql('SELECT count(*) AS n FROM timeline_variables')['n'][0])
display(tl.sql('SELECT dtype, count(*) AS n FROM timeline_variables GROUP BY dtype ORDER BY n DESC'))
tl.search('cell voltage', limit=8)

catalog size: 1405


dtype,n
str,i64
"""float""",575
"""bool""",453
"""unknown""",207
"""int""",170


var_key,dtype,description,n_sessions,first_seen,last_seen
str,str,str,i64,date,date
"""pcm.regen.maxCellVoltage""","""float""","""Regen Max Cell Voltage""",289,2026-05-01,2026-05-31
"""bms.stack.cells.cellV[0]""","""float""","""Cell Voltage""",248,2026-05-01,2026-05-29
"""bms.stack.cells.cellV[100]""","""float""","""Cell Voltage""",248,2026-05-01,2026-05-29
"""bms.stack.cells.cellV[101]""","""float""","""Cell Voltage""",248,2026-05-01,2026-05-29
"""bms.stack.cells.cellV[106]""","""float""","""Cell Voltage""",248,2026-05-01,2026-05-29
"""bms.stack.cells.cellV[107]""","""float""","""Cell Voltage""",248,2026-05-01,2026-05-29
"""bms.stack.cells.cellV[108]""","""float""","""Cell Voltage""",248,2026-05-01,2026-05-29
"""bms.stack.cells.cellV[109]""","""float""","""Cell Voltage""",248,2026-05-01,2026-05-29


Note how much of the catalog is `bool` and `int`. The car's telemetry is overwhelmingly discrete — fault flags, states, counters — and it is logged on change rather than at a fixed rate. That combination is why the columnstore gets it down to roughly 4 bytes per sample, and also why a naive "downsample to 1 Hz" tier would save nothing: most variables already average only a couple of samples per second.

## 2. The actual question

> *"Find me incidents over all testing sessions in May where the minimum cell voltage sagged."*

This is answered entirely from `session_var_stats`. **Zero raw samples are read.**

In [3]:
VAR = 'bms.stack.mma.cellV.min'

started = time.perf_counter()
hits = tl.find(VAR, below=3.2, month='2026-05', min_samples=100)
elapsed = (time.perf_counter() - started) * 1000

print(f'{hits.height} sessions matched in {elapsed:.1f} ms')
hits.head(12)

12 sessions matched in 15.2 ms


test_day,start_utc,session_id,source_key,v_min,v_max,v_mean,n,n_invalid
date,"datetime[μs, UTC]",i64,str,f64,f64,f64,i64,i64
2026-05-02,2026-05-02 12:22:10 UTC,20,"""REV 11/05-02/2ndMay08-22-10.csv""",0.0,0.0,0.0,118,0
2026-05-18,2026-05-18 19:41:18 UTC,214,"""REV 11/05-18/18thMay15-41-18.csv""",0.0,0.0,0.0,353,0
2026-05-26,2026-05-26 17:21:14 UTC,263,"""REV 11/05-26/26thMay13-21-14.csv""",0.0,3.8203,3.733424,16734,0
2026-05-01,2026-05-01 19:58:15 UTC,18,"""REV 11/05-01/1stMay15-58-15.csv""",1.5802,3.3532,3.138647,2107,0
2026-05-12,2026-05-12 15:42:20 UTC,111,"""REV 11/05-12/12thMay11-42-20.csv""",1.6529,3.7818,3.623739,15869,0
2026-05-14,2026-05-14 19:05:18 UTC,185,"""REV 11/05-14/14thMay15-05-18.csv""",1.771,3.7668,3.243424,22164,0
2026-05-02,2026-05-02 17:52:38 UTC,49,"""REV 11/05-02/2ndMay13-52-38.csv""",1.9062,3.6998,3.462445,3911,0
2026-05-01,2026-05-01 17:54:17 UTC,11,"""REV 11/05-01/1stMay13-54-17.csv""",2.9355,3.7935,3.545028,20405,0
2026-05-02,2026-05-02 17:59:04 UTC,50,"""REV 11/05-02/2ndMay13-59-04.csv""",2.9551,2.9949,2.977746,2357,0


Same shape of question, different signal — nothing about the query needs to know which variable it is, so this generalises to any of the ~1,100 signals the car logs.

In [4]:
started = time.perf_counter()
fast = tl.find('pcm.moc.motor.wheelSpeed', above=15.0, month='2026-05')
print(f'{fast.height} sessions above 15 m/s in {(time.perf_counter()-started)*1000:.1f} ms')
fast.head(8)

42 sessions above 15 m/s in 6.3 ms


test_day,start_utc,session_id,source_key,v_min,v_max,v_mean,n,n_invalid
date,"datetime[μs, UTC]",i64,str,f64,f64,f64,i64,i64
2026-05-18,2026-05-18 19:41:18 UTC,214,"""REV 11/05-18/18thMay15-41-18.csv""",103.0,103.0,103.0,352,0
2026-05-01,2026-05-01 17:54:17 UTC,11,"""REV 11/05-01/1stMay13-54-17.csv""",-0.305999,60.681362,11.093686,20381,0
2026-05-18,2026-05-18 19:03:17 UTC,212,"""REV 11/05-18/18thMay15-03-17.csv""",-0.875498,60.672863,5.208148,12693,0
2026-05-18,2026-05-18 20:22:11 UTC,223,"""REV 11/05-18/18thMay16-22-11.csv""",-0.263499,60.638863,3.703199,6183,0
2026-05-01,2026-05-01 16:56:40 UTC,8,"""REV 11/05-01/1stMay12-56-40.csv""",-1.869996,60.638863,19.426166,11900,0
2026-05-18,2026-05-18 16:12:07 UTC,202,"""REV 11/05-18/18thMay12-12-07.csv""",-0.1785,60.604862,2.869687,17060,0
2026-05-18,2026-05-18 20:51:47 UTC,225,"""REV 11/05-18/18thMay16-51-47.csv""",-0.0935,60.604862,11.375261,1840,0
2026-05-18,2026-05-18 20:35:29 UTC,224,"""REV 11/05-18/18thMay16-35-29.csv""",-0.17,60.579361,5.40904,2707,0


## 3. What a season-wide view finds that a single log cannot

Every value in a PER log is written into a float32 field. When firmware writes a raw integer instead of a measurement, the bit pattern decodes as a tiny denormal.

`pcm.pedals.accel` emits `2.8026e-45` — bit pattern `2` — to mean *pedal reading invalid*. That value **looks like zero to every naive aggregate**, so an invalid pedal silently reads as "pedal not pressed".

The timeline flags these as `n_invalid` and excludes them from every statistic.

In [5]:
report = tl.invalid_report(min_fraction=0.001)
report.select(['var_key', 'dtype', 'sentinel_bits', 'invalid_fraction',
               'sessions_affected', 'sessions_total', 'n_invalid'])

var_key,dtype,sentinel_bits,invalid_fraction,sessions_affected,sessions_total,n_invalid
str,str,i64,f64,i64,i64,"decimal[38,0]"
"""pcm.pedals.accel""","""float""",8277800,0.156138,135,289,1284910
"""bms.pack.capacity""","""int""",0,0.002021,6,248,1580
"""bms.stack.mma.heatsinkTemp.max""","""float""",6000000,0.001971,5,248,1543
"""pcm.tractionControl.smo.fr""","""float""",8339694,0.001062,6,289,8792


In [6]:
# Per-session breakdown for the pedal: some sessions are entirely invalid.
pedal = tl.stats('pcm.pedals.accel', month='2026-05').with_columns(
    (pl.col('n_invalid') / pl.col('n')).alias('invalid_frac')
)
fig = go.Figure(go.Bar(x=pedal['start_utc'], y=pedal['invalid_frac']))
fig.update_layout(
    title='pcm.pedals.accel — fraction of samples that are the invalid sentinel',
    xaxis_title='session start (UTC)', yaxis_title='invalid fraction',
    height=380, yaxis_tickformat='.0%',
)
fig.show()

## 4. Drilling down to raw samples

Only now do we touch the hypertable — and only for the sessions the summary tier already selected. `load()` returns a PERDA `DataInstance`, so anything downstream (arithmetic, joins, `Analyzer` plotting) works unchanged.

In [7]:
# Among the matching sessions, take the one with the most samples of this
# signal -- the shortest session would make for a thin plot.
target = int(
    hits.sort('n', descending=True)['session_id'][0] if hits.height
    else tl.sql('SELECT session_id FROM timeline_sessions ORDER BY n_rows DESC LIMIT 1')['session_id'][0]
)

started = time.perf_counter()
signal = tl.load(VAR, session_id=target)
print(f'{len(signal.value_np):,} samples in {(time.perf_counter()-started)*1000:.0f} ms')
print(type(signal).__name__, '->', signal.cpp_name)

fig = go.Figure(go.Scatter(x=signal.timestamp_np / 1e6, y=signal.value_np, mode='lines'))
fig.update_layout(
    title=f'{VAR} — session {target}',
    xaxis_title='seconds into session', yaxis_title='volts', height=380,
)
fig.show()

22,164 samples in 59 ms
DataInstance -> bms.stack.mma.cellV.min


## 5. The season at a glance

One row per session, straight from the summary tier — the view that simply did not exist before.

In [8]:
season = tl.stats(VAR, month='2026-05')

fig = go.Figure()
fig.add_trace(go.Scatter(x=season['start_utc'], y=season['v_min'],
                         mode='markers', name='min'))
fig.add_trace(go.Scatter(x=season['start_utc'], y=season['v_p50'],
                         mode='markers', name='median'))
fig.add_trace(go.Scatter(x=season['start_utc'], y=season['v_max'],
                         mode='markers', name='max'))
fig.update_layout(title=f'{VAR} across every May session',
                  xaxis_title='session start (UTC)', yaxis_title='volts', height=420)
fig.show()

## 6. Raw SQL — the surface the LLM layer will target

Three stable views (`v_sessions`, `v_stats`, `v_samples`) hide ids, epoch microseconds and partition layout. A text-to-SQL model writes against these names and never needs to know the storage design.

In [9]:
tl.sql('''
    SELECT test_day,
           count(DISTINCT session_id) AS sessions,
           round(max(v_max) FILTER (WHERE var_key = 'bms.pack.current')::numeric, 1)
               AS peak_pack_current,
           round(min(v_min) FILTER (WHERE var_key = 'bms.stack.mma.cellV.min')::numeric, 3)
               AS lowest_cell_v
    FROM v_stats
    WHERE var_key IN ('bms.pack.current', 'bms.stack.mma.cellV.min')
    GROUP BY test_day
    ORDER BY test_day
''')

test_day,sessions,peak_pack_current,lowest_cell_v
date,i64,"decimal[38,1]","decimal[38,3]"
2026-05-01,18,189.3,1.580
2026-05-02,32,155.3,0.000
2026-05-11,20,7.1,4.115
2026-05-12,21,114.8,1.653
2026-05-13,34,128.8,3.744
2026-05-14,20,143.2,1.771
2026-05-18,42,185.7,0.000
2026-05-20,26,176.8,3.438
2026-05-26,10,119.3,0.000


In [10]:
# Which variables were most *active* (value changes per second) across the month?
tl.sql('''
    SELECT var_key, dtype,
           round(avg(hz)::numeric, 1)                     AS avg_hz,
           sum(n_changes)                                  AS total_changes,
           round(max(max_gap_us)/1e6::numeric, 1)          AS worst_gap_s
    FROM v_stats
    WHERE dtype = 'bool'
    GROUP BY var_key, dtype
    HAVING sum(n_changes) > 0
    ORDER BY total_changes DESC
    LIMIT 10
''')

var_key,dtype,avg_hz,total_changes,worst_gap_s
str,str,"decimal[38,1]","decimal[38,0]","decimal[38,1]"
"""pcm.digitalInput1""","""bool""",9.2,91219,235.7
"""pcm.digitalInput3""","""bool""",9.2,85670,235.7
"""pcm.digitalInput2""","""bool""",9.2,70501,235.7
"""pcm.digitalInput4""","""bool""",9.2,69458,235.7
"""pcm.bspd.highCurrent""","""bool""",9.2,11924,235.7
"""pcm.pedals.implausibility.anyImplausibility""","""bool""",9.2,8608,235.7
"""bms.faults.tempMonitoringFault""","""bool""",9.4,8574,235.5
"""pcm.bspd.hardBraking""","""bool""",9.2,8536,235.7
"""pcm.pedals.faults.brakeAngleOOB""","""bool""",9.2,8528,235.7


## 7. Storage and speed

The numbers that decide whether this is worth running.

In [11]:
display(tl.sizes())

totals = tl.sql('''
    SELECT count(*) AS sessions, sum(n_rows) AS samples,
           round(sum(source_bytes)/1e9::numeric, 1) AS source_gb
    FROM timeline_sessions
''')
display(totals)

rows = int(totals['samples'][0])
source_bytes = float(totals['source_gb'][0]) * 1e9
compressed = float(tl.sql("SELECT hypertable_size('timeline_samples') AS b")['b'][0])
print(f'\n{rows:,} samples, {compressed/rows:.2f} bytes per sample '
      f'({source_bytes/compressed:.1f}x smaller than the source CSV)')

object,size
str,str
"""samples (hypertable)""","""6737 MB"""
"""timeline_session_var_stats""","""66 MB"""
"""timeline_sessions""","""272 kB"""
"""timeline_variables""","""5008 kB"""


sessions,samples,source_gb
i64,"decimal[38,0]","decimal[38,1]"
298,1681854727,47.4



1,681,854,727 samples, 4.20 bytes per sample (6.7x smaller than the source CSV)


In [12]:
def timed(fn, repeats=3):
    runs = []
    for _ in range(repeats):
        started = time.perf_counter()
        fn()
        runs.append((time.perf_counter() - started) * 1000)
    return min(runs)

day = tl.sql(
    'SELECT test_day FROM timeline_sessions WHERE session_id = %s', (target,)
)['test_day'][0]

cases = {
    'threshold scan (summary tier)':
        lambda: tl.find(VAR, below=3.2, month='2026-05'),
    'per-session stats, one month':
        lambda: tl.stats(VAR, month='2026-05'),
    'one variable, one session (raw)':
        lambda: tl.samples(VAR, session_id=target),
    'one variable, one whole test day':
        lambda: tl.samples(VAR, test_day=day),
    'full raw scan (no pruning possible)':
        lambda: tl.sql('SELECT count(*), avg(value) FROM timeline_samples'),
}

for label, fn in cases.items():
    print(f'{label:36s} {timed(fn):8.1f} ms')

threshold scan (summary tier)             4.6 ms
per-session stats, one month              8.0 ms
one variable, one session (raw)          43.0 ms


one variable, one whole test day        169.6 ms


full raw scan (no pruning possible)    4762.0 ms


---

### Where this goes next

1. **Backfill the rest of REV 11** — February through June, ~1,865 logs / 218 GB. Ingest runs at roughly 150k samples/s per worker and the store lands near 25 GB.
2. **Hook it to the ingest pipeline** — one more Celery task after `autotag`, so new logs join the timeline automatically.
3. **Point a text-to-SQL model at the three views**, with the variable catalog as retrieval context.
4. **Cross-revision aliasing** (`ams.*` → `bms.*`) if the 2022–2025 logs are worth pulling in — the only piece that needs hand-verification.

Open design question: whether `perda.timeline` stays a client to a shared TimescaleDB, or also ships a self-contained reader so a teammate can use it without a server.